In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =============================================================================
# 1. PHYSICAL MODEL PARAMETERS
# =============================================================================
vp = [1000.0, 3350.0, 4750.0]        # P-wave velocities (m/s)
thicknesses = [10.0, 25.0, 45.0]     # Layer thicknesses (m)
densities = [2000.0, 2275.0, 2450.0] # Layer densities (kg/m3)

# Calculate discrete interface depths
# Layer 1: 0 to 10m | Layer 2: 10 to 35m | Layer 3: 35 to 80m
z_nodes = [0.0, thicknesses[0], thicknesses[0] + thicknesses[1], sum(thicknesses)]

# Profile configuration
src_x = -120.0 # Source fixed at x = -120m
rec_small = -70.0 # Small offset receiver (50m away)
rec_large = 100.0 # Large offset receiver (220m away)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1, 2.5]})

# =============================================================================
# 2. CORRECTED 1D VELOCITY MODEL PLOT
# =============================================================================
# Build discrete step blocks for proper blocky 1D visualization
v_plot = [vp[0], vp[0], vp[1], vp[1], vp[2], vp[2]]
z_plot = [z_nodes[0], z_nodes[1], z_nodes[1], z_nodes[2], z_nodes[2], z_nodes[3]]

ax1.plot(v_plot, z_plot, color='black', linewidth=2.5, label='Vp Profile')
ax1.set_xlim(0, 5500)
ax1.set_ylim(z_nodes[-1], 0) # Invert depth axis (0 at top)
ax1.set_xlabel('P-Wave Velocity (m/s)', fontweight='bold')
ax1.set_ylabel('Depth (m)', fontweight='bold')
ax1.set_title('1D Velocity Model Profile', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.6)

# Annotate layer properties inside the step plot
for i in range(3):
    z_mid = (z_nodes[i] + z_nodes[i+1]) / 2
    ax1.text(vp[i] - 200, z_mid, f"Layer {i+1}\n{vp[i]} m/s\n{densities[i]} kg/m³", 
             ha='right', va='center', fontsize=9, bbox=dict(facecolor='white', alpha=0.7, boxstyle='round'))

# =============================================================================
# 3. EXACT SNELL'S LAW RAYPATH TRACING
# =============================================================================
# Draw background horizontal interface lines
for depth in z_nodes:
    ax2.axhline(y=depth, color='gray', linestyle='--', alpha=0.7)

# Label layers in the cross-section space
ax2.text(-145, 5, "Layer 1", color='gray', fontweight='bold')
ax2.text(-145, 22, "Layer 2", color='gray', fontweight='bold')
ax2.text(-145, 55, "Layer 3", color='gray', fontweight='bold')

# --- RAY 1: Small Offset Reflection (Reflects off Interface 1 at 10m depth) ---
mid_x_small = (src_x + rec_small) / 2
ax2.plot([src_x, mid_x_small, rec_small], [0, z_nodes[1], 0], 
         color='crimson', linewidth=2, linestyle='-', label='Small Offset Reflection (Interface 1)')

# --- RAY 2: Large Offset Head Wave (Travels along Bottom Interface 2 at 35m depth) ---
# Ray parameter for critical refraction along top of Layer 3: p = 1 / V3
p_head = 1.0 / vp[2]
# Solve transmission angles using Snell's Law: sin(theta) = p * V
theta1 = np.arcsin(p_head * vp[0])
theta2 = np.arcsin(p_head * vp[1]) # This is exactly the critical angle between 2 and 3

# Compute horizontal distance components traversed while moving vertically through layers
dx1 = thicknesses[0] * np.tan(theta1)
dx2 = thicknesses[1] * np.tan(theta2)
x_crit_entry = dx1 + dx2 # Horizontal distance to reach bottom interface

# Total horizontal distance from source to receiver
total_distance = rec_large - src_x
x_interface_travel = total_distance - (2 * x_crit_entry)

if x_interface_travel > 0:
    # Coordinate nodes for the head wave raypath
    ray2_x = [
        src_x, 
        src_x + dx1, 
        src_x + x_crit_entry, 
        src_x + x_crit_entry + x_interface_travel, 
        src_x + x_crit_entry + x_interface_travel + dx2, 
        rec_large
    ]
    ray2_y = [0, z_nodes[1], z_nodes[2], z_nodes[2], z_nodes[1], 0]
    
    ax2.plot(ray2_x, ray2_y, color='royalblue', linewidth=2.5, label='Large Offset Head Wave (Interface 2)')
    # Highlight the head wave segment running along the boundary
    ax2.plot([src_x + x_crit_entry, src_x + x_crit_entry + x_interface_travel], [z_nodes[2], z_nodes[2]], 
             color='gold', linewidth=4, linestyle='-', label='Head Wave Boundary Path')

# =============================================================================
# 4. PLOT EMBELLISHMENTS & DISPLAY
# =============================================================================
# Plot geometry locations
ax2.plot(src_x, 0, marker='*', color='red', markersize=14, linestyle='None', label='Source')
ax2.plot(rec_small, 0, marker='^', color='green', markersize=10, linestyle='None', label='Small-offset Rec')
ax2.plot(rec_large, 0, marker='^', color='darkorange', markersize=10, linestyle='None', label='Large-offset Rec')

ax2.set_xlim(-150, 150)
ax2.set_ylim(80, -5) # Matches cumulative thickness
ax2.set_xlabel('Profile Position (m)', fontweight='bold')
ax2.set_ylabel('Depth (m)', fontweight='bold')
ax2.set_title("Snell's Law Raypaths & Head Wave Propagation", fontsize=12, fontweight='bold')
ax2.legend(loc='lower right', fontsize=9)
ax2.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# =============================================================================
# 1. PHYSICAL MODEL PARAMETERS WITH 2D CAVE
# =============================================================================
vp_layers = [1000.0, 3350.0, 4750.0]  # Layer velocities (m/s)
thicknesses = [10.0, 25.0, 45.0]     # Layer thicknesses (m)
z_nodes = [0.0, 10.0, 35.0, 80.0]     # Interface depths (m)

# Cave parameters (Centered at x=0, z=22.5)
cave_v = 1500.0                       # P-wave velocity of cave (m/s)
cave_w, cave_h = 20.0, 20.0          # Width and height (m)
cave_x1, cave_x2 = -10.0, 10.0        # Left and right bounds
cave_z1, cave_z2 = 12.5, 32.5         # Top and bottom bounds (inside Layer 2)

# Geometry
src_x = -120.0
rec_far = 120.0
rec_near = 40.0

fig, ax = plt.subplots(figsize=(12, 7))

# =============================================================================
# 2. DRAW 2D VELOCITY MODEL GRID
# =============================================================================
# Background layers
ax.add_patch(patches.Rectangle((-150, 0), 300, 10, facecolor='#eaeaea', edgecolor='gray', linestyle='--'))
ax.add_patch(patches.Rectangle((-150, 10), 300, 25, facecolor='#dcdcdc', edgecolor='gray', linestyle='--'))
ax.add_patch(patches.Rectangle((-150, 35), 300, 45, facecolor='#c8c8c8', edgecolor='gray', linestyle='--'))

# The Cave Patch
cave_patch = patches.Rectangle((cave_x1, cave_z1), cave_w, cave_h, 
                               facecolor='#6baed6', edgecolor='black', linewidth=2, hatch='//',
                               label=f'Cave ({cave_v} m/s)')
ax.add_patch(cave_patch)

# Text labels for materials
ax.text(-140, 5, f"Layer 1: {vp_layers[0]} m/s", weight='bold', color='#333333')
ax.text(-140, 18, f"Layer 2: {vp_layers[1]} m/s", weight='bold', color='#333333')
ax.text(-140, 55, f"Layer 3: {vp_layers[2]} m/s", weight='bold', color='#333333')
ax.text(0, 22.5, f"Cave\n{cave_v} m/s", ha='center', va='center', weight='bold', color='white',
        path_effects=[plt.cm.colors.Normalize()])

# =============================================================================
# 3. TRACE RAYS INTERACTING WITH THE CAVE (SNELL'S LAW)
# =============================================================================

# --- RAY 1: Reflection off the Top of the Cave ---
# Path: Source -> Layer 1 boundary -> Cave top corner/edge -> Surface
# Midpoint for reflection off the flat top surface of the cave
mid_x_ref = (src_x + rec_near) / 2 # -40m, hits the top boundary if we extend it
# Let's target the exact top-left corner of the cave (-10, 12.5)
ax.plot([src_x, -10, rec_near], [0, cave_z1, 0], 
        color='crimson', linewidth=2.5, linestyle='-', label='Cave Top Reflection')

# --- RAY 2: Transmission through the Cave (Slowing down & Bending) ---
# When entering the cave from Layer 2 (3350 m/s) to Cave (1500 m/s), 
# the velocity drops, meaning the ray bends *towards* the normal vector.
# Let's simulate a ray shooting from the source into the cave's left wall, 
# refracting downward, and exiting through the bottom/right side.

# Node coords: Surface -> Layer 1 interface -> Cave Left Wall -> Cave Right Wall -> Layer 1 interface -> Receiver
ray_cave_x = [src_x, -50.0, cave_x1, cave_x2, 60.0, rec_far]
ray_cave_y = [0.0, 10.0, 20.0, 25.0, 10.0, 0.0]

ax.plot(ray_cave_x, ray_cave_y, color='darkorange', linewidth=2.5, 
        marker='o', markersize=4, label='Ray Transmitted Through Cave')

# --- RAY 3: Standard Head Wave (Unperturbed reference below the cave) ---
p_head = 1.0 / vp_layers[2]
theta1 = np.arcsin(p_head * vp_layers[0])
theta2 = np.arcsin(p_head * vp_layers[1])
dx1 = thicknesses[0] * np.tan(theta1)
dx2 = thicknesses[1] * np.tan(theta2)
x_crit = dx1 + dx2
x_interface_travel = (rec_far - src_x) - (2 * x_crit)

ray3_x = [src_x, src_x + dx1, src_x + x_crit, src_x + x_crit + x_interface_travel, src_x + x_crit + x_interface_travel + dx2, rec_far]
ray3_y = [0, 10, 35, 35, 10, 0]
ax.plot(ray3_x, ray3_y, color='teal', linewidth=2, linestyle='--', alpha=0.7, label='Basement Head Wave (Deep)')

# =============================================================================
# 4. DISPLAY AND COSMETICS
# =============================================================================
ax.plot(src_x, 0, marker='*', color='red', markersize=15, linestyle='None', label='Source')
ax.plot(rec_near, 0, marker='^', color='green', markersize=10, linestyle='None', label='Near Receiver')
ax.plot(rec_far, 0, marker='^', color='blue', markersize=10, linestyle='None', label='Far Receiver')

ax.set_xlim(-150, 150)
ax.set_ylim(70, -2) # Inverted depth axis
ax.set_xlabel('Profile Position (m)', fontweight='bold')
ax.set_ylabel('Depth (m)', fontweight='bold')
ax.set_title("2D Velocity Model with Low-Velocity Cave anomalies", fontsize=14, fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# =============================================================================
# 1. PHYSICAL MODEL PARAMETERS WITH 2D CAVE
# =============================================================================
vp_layers = [1000.0, 3350.0, 4750.0]  # Layer velocities (m/s)
thicknesses = [10.0, 25.0, 45.0]     # Layer thicknesses (m)
z_nodes = [0.0, 10.0, 35.0, 80.0]     # Interface depths (m)

# Cave parameters (Centered at x=0, z=22.5)
cave_v = 1500.0                       # P-wave velocity of cave (m/s)
cave_w, cave_h = 20.0, 20.0          # Width and height (m)
cave_x1, cave_x2 = -10.0, 10.0        # Left and right bounds
cave_z1, cave_z2 = 12.5, 32.5         # Top and bottom bounds (inside Layer 2)

# Geometry
src_x = -120.0
receivers = np.linspace(-150, 150, 301)  # High resolution receiver array

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 10), sharex=True)

# =============================================================================
# 2. DRAW 2D VELOCITY MODEL & RAYS (AX1)
# =============================================================================
# Background layers
ax1.add_patch(patches.Rectangle((-150, 0), 300, 10, facecolor='#eaeaea', edgecolor='gray', linestyle='--'))
ax1.add_patch(patches.Rectangle((-150, 10), 300, 25, facecolor='#dcdcdc', edgecolor='gray', linestyle='--'))
ax1.add_patch(patches.Rectangle((-150, 35), 300, 45, facecolor='#c8c8c8', edgecolor='gray', linestyle='--'))

# The Cave Patch
cave_patch = patches.Rectangle((cave_x1, cave_z1), cave_w, cave_h, 
                               facecolor='#6baed6', edgecolor='black', linewidth=2, hatch='//',
                               label=f'Cave ({cave_v} m/s)')
ax1.add_patch(cave_patch)

# Text labels for materials
ax1.text(-140, 5, f"Layer 1: {vp_layers[0]} m/s", weight='bold', color='#333333')
ax1.text(-140, 18, f"Layer 2: {vp_layers[1]} m/s", weight='bold', color='#333333')
ax1.text(-140, 55, f"Layer 3: {vp_layers[2]} m/s", weight='bold', color='#333333')
# FIXED: Removed the buggy path_effects mapping
ax1.text(0, 22.5, f"Cave\n{cave_v} m/s", ha='center', va='center', weight='bold', color='black')

# --- TRACE EXEMPLAR RAYS ---
# Ray A: Small offset reflection
ax1.plot([src_x, -75, -30], [0, 10, 0], color='crimson', linewidth=2, label='Layer 1 Reflection')

# Ray B: Ray passing through the cave center
ray_cave_x = [src_x, -50.0, cave_x1, cave_x2, 60.0, 120.0]
ray_cave_y = [0.0, 10.0, 22.5, 22.5, 10.0, 0.0]
ax1.plot(ray_cave_x, ray_cave_y, color='darkorange', linewidth=2.5, marker='o', label='Ray Through Cave')

# Ray C: Unperturbed basement head wave
p_head = 1.0 / vp_layers[2]
theta1 = np.arcsin(p_head * vp_layers[0])
theta2 = np.arcsin(p_head * vp_layers[1])
dx1 = thicknesses[0] * np.tan(theta1)
dx2 = thicknesses[1] * np.tan(theta2)
x_crit = dx1 + dx2
x_interface_travel = (140.0 - src_x) - (2 * x_crit)
ray3_x = [src_x, src_x + dx1, src_x + x_crit, src_x + x_crit + x_interface_travel, src_x + x_crit + x_interface_travel + dx2, 140.0]
ray3_y = [0.0, 10.0, 35.0, 35.0, 10.0, 0.0]
ax1.plot(ray3_x, ray3_y, color='teal', linewidth=2, linestyle='--', label='Basement Head Wave')

ax1.plot(src_x, 0, marker='*', color='red', markersize=15, label='Source')
ax1.set_ylim(75, -2)
ax1.set_ylabel('Depth (m)', fontweight='bold')
ax1.set_title("2D Velocity Model & Raypaths", fontsize=12, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.5)
ax1.legend(loc='lower right')

# =============================================================================
# 3. CALCULATE AND PLOT TRAVEL-TIME (T-X) CURVES (AX2)
# =============================================================================
t_direct = []
t_head_layer3 = []

# Analytically calculate times for every surface receiver position x
for rx in receivers:
    distance = np.abs(rx - src_x)
    
    # 1. Direct Wave (traveling horizontally in layer 1)
    t_direct.append(distance / vp_layers[0])
    
    # 2. Layer 3 Head Wave (Critical Refraction time)
    # T_head = distance/V3 + intercept_time
    if distance >= 2 * x_crit:
        t_intercept = 2 * (thicknesses[0] * np.cos(theta1) / vp_layers[0] + 
                           thicknesses[1] * np.cos(theta2) / vp_layers[1])
        t_total = (distance / vp_layers[2]) + t_intercept
        
        # Apply travel-time delay shadow if the ray passes underneath the cave projection
        # High velocity layer 2 is missing 20m of width, replaced by low velocity air/cave
        # Delay delta_t = width * (1/V_cave - 1/V_layer2)
        if rx > cave_x2 + 30: # Receiver is past the shadow zone of downcoming rays
            t_delay = cave_w * ((1.0 / cave_v) - (1.0 / vp_layers[1]))
            t_total += t_delay
            
        t_head_layer3.append(t_total)
    else:
        t_head_layer3.append(np.nan)

# Plot curves
ax2.plot(receivers, np.array(t_direct) * 1000, color='magenta', linewidth=2, label='Direct Wave (Layer 1)')
ax2.plot(receivers, np.array(t_head_layer3) * 1000, color='teal', linewidth=2.5, label='Layer 3 Head Wave')

# Highlight the low-velocity anomaly delay (pull-up delay step)
ax2.axvspan(cave_x2 + 20, 150, color='orange', alpha=0.15, label='Cave Shadow Zone (Time Delay)')
ax2.text(70, 95, "Low-Velocity Delay\n(Travel-time Step up)", color='darkred', weight='bold', ha='center')

ax2.set_xlim(-150, 150)
ax2.set_xlabel('Profile Position (m)', fontweight='bold')
ax2.set_ylabel('Travel Time (ms)', fontweight='bold')
ax2.set_title("Travel-Time (t-x) Curves showing Anomalous Delay Shadow", fontsize=12, fontweight='bold')
ax2.grid(True, linestyle=':', alpha=0.5)
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.optimize import minimize

# =============================================================================
# 1. PHYSICAL MODEL PARAMETERS
# =============================================================================
vp_layers = [2600.0, 3600.0, 4750.0]  # Layer velocities (m/s)
thicknesses = [10.0, 25.0, 45.0]     # Layer thicknesses (m)
z_nodes = [0.0, 10.0, 35.0, 80.0]     # Interface depths (m)

# Cave parameters (Centered at x=0, z=22.5)
cave_v = 1500.0                       
cave_w, cave_h = 20.0, 20.0          
cave_x1, cave_x2 = -10.0, 10.0        
cave_z1, cave_z2 = 12.5, 32.5         

# Geometry setup
src_x = -150.0
rec_near = 40.0
rec_far = 140.0      # MODIFIED: Shifted receiver closer to center to force a mid-cave path
receivers = np.linspace(-140, 140, 561)

# =============================================================================
# 2. EXACT SNELL'S LAW SOLVERS VIA FERMAT'S PRINCIPLE
# =============================================================================
def solve_reflection_l1(src_x, rec_x):
    """Finds the reflection point on Layer 1 interface (z=10) via Fermat's Principle."""
    def t_field(x_int):
        return (np.sqrt((x_int - src_x)**2 + 10**2) + np.sqrt((rec_x - x_int)**2 + 10**2)) / vp_layers[0]
    res = minimize(t_field, x0=[(src_x + rec_x)/2], bounds=[(min(src_x, rec_x), max(src_x, rec_x))])
    return res.x

def solve_cave_ray(src_x, rec_x):
    """
    Finds intersection points to satisfy Snell's Law through vertical cave walls.
    Optimizes:
      x1: Entry into Layer 2 at z=10
      z2: Entry into Cave Left Wall at x=-10
      z3: Exit from Cave Right Wall at x=10
      x4: Exit from Layer 2 back into Layer 1 at z=10
    """
    def t_total(p):
        x1, z2, z3, x4 = p
        # Path 1: Source to Layer 1 Interface
        t1 = np.sqrt((x1 - src_x)**2 + 10**2) / vp_layers[0]
        # Path 2: Layer 1 Interface to Cave Left Wall (x = -10)
        t2 = np.sqrt((-10 - x1)**2 + (z2 - 10)**2) / vp_layers[1]
        # Path 3: Through the Cave (From x = -10 to x = 10)
        t3 = np.sqrt((10 - (-10))**2 + (z3 - z2)**2) / cave_v
        # Path 4: Cave Right Wall (x = 10) to Layer 1 Interface
        t4 = np.sqrt((x4 - 10)**2 + (10 - z3)**2) / vp_layers[1]
        # Path 5: Layer 1 Interface to Far Receiver
        t5 = np.sqrt((rec_x - x4)**2 + 10**2) / vp_layers[0]
        return t1 + t2 + t3 + t4 + t5

    # Initial guesses and bounds constrained within the physical cave vertical limits
    x0 = [-60.0, 22.5, 22.5, 40.0]
    bnds = [(src_x, -10.0), (cave_z1, cave_z2), (cave_z1, cave_z2), (10.0, rec_x)]
    res = minimize(t_total, x0=x0, bounds=bnds)
    return res.x

# Calculate exact unperturbed Head Wave geometries
p_head = 1.0 / vp_layers[2]
theta1 = np.arcsin(p_head * vp_layers[0])
theta2 = np.arcsin(p_head * vp_layers[1])
dx1 = thicknesses[0] * np.tan(theta1)
dx2 = thicknesses[1] * np.tan(theta2)
x_crit = dx1 + dx2

# =============================================================================
# 3. FIGURES 1 & 2: WITH CAVE ENVIRONMENT (MODEL & TRAVEL TIME)
# =============================================================================
fig1, ax1 = plt.subplots(figsize=(11, 6))

# Background Layers
ax1.add_patch(patches.Rectangle((-150, 0), 300, 10, facecolor='#eaeaea', edgecolor='gray', linestyle='--'))
ax1.add_patch(patches.Rectangle((-150, 10), 300, 25, facecolor='#dcdcdc', edgecolor='gray', linestyle='--'))
ax1.add_patch(patches.Rectangle((-150, 35), 300, 45, facecolor='#c8c8c8', edgecolor='gray', linestyle='--'))
cave_patch = patches.Rectangle((cave_x1, cave_z1), cave_w, cave_h, facecolor='#6baed6', edgecolor='black', linewidth=2, hatch='//', label=f'Cave ({cave_v} m/s)')
ax1.add_patch(cave_patch)

ax1.text(-140, 5, f"Layer 1: {vp_layers[0]} m/s", weight='bold')
ax1.text(-140, 18, f"Layer 2: {vp_layers[1]} m/s", weight='bold')
ax1.text(-140, 55, f"Layer 3: {vp_layers[2]} m/s", weight='bold')
ax1.text(0, 22.5, f"Cave\n{cave_v} m/s", ha='center', va='center', weight='bold')

# --- Ray Tracing Plots ---
# RAY 1: Layer 1 Reflection (Crimson)
x_ref1 = solve_reflection_l1(src_x, rec_near)
ax1.plot([src_x, x_ref1[0], rec_near], [0.0, 10.0, 0.0], color='crimson', linewidth=2.5, label='Layer 1 Reflection')

# RAY 2: Physics-Enforced Ray Deflected through Cave (Darkorange)
x1_c, z2_c, z3_c, x4_c = solve_cave_ray(src_x, rec_far)
ray_c_x = [src_x, x1_c, cave_x1, cave_x2, x4_c, rec_far]
ray_c_y = [0.0, 10.0, z2_c, z3_c, 10.0, 0.0]
ax1.plot(ray_c_x, ray_c_y, color='darkorange', linewidth=2.5, marker='o', label='Deflected Ray Through Cave')

# RAY 3: Basement Head Wave (Teal)
x_travel = (rec_far - src_x) - (2 * x_crit)
ray3_x = [src_x, src_x + dx1, src_x + x_crit, src_x + x_crit + x_travel, src_x + x_crit + x_travel + dx2, rec_far]
ray3_y = [0.0, 10.0, 35.0, 35.0, 10.0, 0.0]
ax1.plot(ray3_x, ray3_y, color='teal', linewidth=2, linestyle='--', label='Basement Head Wave')

ax1.plot(src_x, 0, marker='*', color='red', markersize=15, linestyle='None', label='Source')
ax1.plot(rec_far, 0, marker='^', color='blue', markersize=10, linestyle='None', label='Far Receiver')

ax1.set_xlim(-150, 150)
ax1.set_ylim(75, -2)
ax1.set_xlabel('Profile Position (m)', fontweight='bold')
ax1.set_ylabel('Depth (m)', fontweight='bold')
ax1.set_title("Figure 1: 2D Velocity Model with True Snell's Law Deflections", fontsize=12, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, linestyle=':', alpha=0.5)

# FIGURE 2: Travel-Time Curves (With Cave)
fig2, ax2 = plt.subplots(figsize=(11, 5))
t_direct, t_head = [], []
for rx in receivers:
    dist = np.abs(rx - src_x)
    t_direct.append(dist / vp_layers[0])
    if dist >= 2 * x_crit:
        t_base = (dist / vp_layers[2]) + 2 * (thicknesses[0]*np.cos(theta1)/vp_layers[0] + thicknesses[1]*np.cos(theta2)/vp_layers[1])
        if rx > cave_x2 + 25:
            t_base += cave_w * ((1.0 / cave_v) - (1.0 / vp_layers[1]))
        t_head.append(t_base)
    else:
        t_head.append(np.nan)

ax2.plot(receivers, np.array(t_direct)*1000, color='crimson', linewidth=2, label='Direct Wave')
ax2.plot(receivers, np.array(t_head)*1000, color='teal', linewidth=2.5, label='Layer 3 Head Wave')
ax2.axvspan(cave_x2 + 20, 150, color='darkorange', alpha=0.1, label='Cave Delay Shadow Zone')
ax2.set_xlim(-150, 150)
ax2.set_ylabel('Travel Time (ms)', fontweight='bold')
ax2.set_title("Figure 2: Travel-Time (t-x) Curves (With Cave Delay)", fontsize=12, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, linestyle=':')

# =============================================================================
# 4. FIGURES 3 & 4: REFERENCE BACKGROUND PLOTS (WITHOUT CAVE)
# =============================================================================
fig3, ax3 = plt.subplots(figsize=(11, 6))
ax3.add_patch(patches.Rectangle((-150, 0), 300, 10, facecolor='#eaeaea', edgecolor='gray', linestyle='--'))
ax3.add_patch(patches.Rectangle((-150, 10), 300, 25, facecolor='#dcdcdc', edgecolor='gray', linestyle='--'))
ax3.add_patch(patches.Rectangle((-150, 35), 300, 45, facecolor='#c8c8c8', edgecolor='gray', linestyle='--'))

ax3.text(-140, 5, f"Layer 1: {vp_layers[0]} m/s", weight='bold')
ax3.text(-140, 18, f"Layer 2: {vp_layers[1]} m/s", weight='bold')
ax3.text(-140, 55, f"Layer 3: {vp_layers[2]} m/s", weight='bold')

# Unperturbed paths (No Cave)
ax3.plot([src_x, x_ref1[0], rec_near], [0.0, 10.0, 0.0], color='crimson', linewidth=2.5, label='Layer 1 Reflection')
ax3.plot(ray3_x, ray3_y, color='teal', linewidth=2, linestyle='--', label='Basement Head Wave')

# Calculate unperturbed Layer 2 reference ray replacing the cave ray
def solve_layer2_ray(src_x, rec_x, target_z):
    def t_l2(p):
        x1, x2 = p
        return (np.sqrt((x1 - src_x)**2 + 10**2)/vp_layers[0] + 
                np.sqrt((x2 - x1)**2 + (target_z - 10)**2 * 2)/vp_layers[1] + 
                np.sqrt((rec_x - x2)**2 + 10**2)/vp_layers[0])
    res = minimize(t_l2, x0=[-40, 40], bounds=[(src_x, 0), (0, rec_x)])
    return res.x

x1_noc, x2_noc = solve_layer2_ray(src_x, rec_far, 22.5)
ax3.plot([src_x, x1_noc, 0.0, x2_noc, rec_far], [0.0, 10.0, 22.5, 10.0, 0.0], color='darkorange', linewidth=2.5, label='Unperturbed Layer 2 Ray')

ax3.plot(src_x, 0, marker='*', color='red', markersize=15, linestyle='None', label='Source')
ax3.plot(rec_far, 0, marker='^', color='blue', markersize=10, linestyle='None', label='Far Receiver')
ax3.set_xlim(-150, 150)
ax3.set_ylim(75, -2)
ax3.set_xlabel('Profile Position (m)', fontweight='bold')
ax3.set_ylabel('Depth (m)', fontweight='bold')
ax3.set_title("Figure 3: Reference Model (Without Cave Anomaly)", fontsize=12, fontweight='bold')
ax3.legend(loc='lower right')
ax3.grid(True, linestyle=':', alpha=0.5)

# FIGURE 4: Reference Travel-Time Curves (Without Cave)
fig4, ax4 = plt.subplots(figsize=(11, 5))
t_head_nocave = []
for rx in receivers:
    dist = np.abs(rx - src_x)
    if dist >= 2 * x_crit:
        t_base = (dist / vp_layers[2]) + 2 * (thicknesses[0]*np.cos(theta1)/vp_layers[0] + thicknesses[1]*np.cos(theta2)/vp_layers[1])
        t_head_nocave.append(t_base)
    else:
        t_head_nocave.append(np.nan)

ax4.plot(receivers, np.array(t_direct)*1000, color='crimson', linewidth=2, label='Direct Wave')
ax4.plot(receivers, np.array(t_head_nocave)*1000, color='teal', linewidth=2.5, label='Layer 3 Head Wave (No Delay)')
ax4.set_xlim(-150, 150)
ax4.set_ylabel('Travel Time (ms)', fontweight='bold')
ax4.set_title("Figure 4: Reference Travel-Time Curves (Linear Baseline)", fontsize=12, fontweight='bold')
ax4.legend(loc='upper left')
ax4.grid(True, linestyle=':')

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.optimize import minimize

# =============================================================================
# 1. PHYSICAL MODEL PARAMETERS
# =============================================================================
vp_layers = [1000.0, 3350.0, 4750.0]  
thicknesses = [10.0, 25.0, 45.0]     
z_nodes = [0.0, 10.0, 35.0, 80.0]     

# Cave parameters (Centered at x=0, z=22.5)
cave_v = 1500.0                       
cave_w, cave_h = 20.0, 20.0          
cave_x1, cave_x2 = -10.0, 10.0        
cave_z1, cave_z2 = 12.5, 32.5         

# Geometry setup
src_x = -120.0
rec_near = 40.0
rec_far = 75.0      
receivers = np.linspace(-150, 150, 301)

# =============================================================================
# 2. FIXED SNELL'S LAW SOLVER (BOUNDARY-VALUE SPECIFIC)
# =============================================================================
def solve_reflection_l1(src_x, rec_x):
    def t_field(x_int):
        return (np.sqrt((x_int - src_x)**2 + 10**2) + np.sqrt((rec_x - x_int)**2 + 10**2)) / vp_layers[0]
    res = minimize(t_field, x0=[(src_x + rec_x)/2], bounds=[(min(src_x, rec_x), max(src_x, rec_x))])
    return res.x

def solve_forced_mid_cave_ray(src_x, rec_x, target_z2):
    """
    Locks the entry point exactly at a specified depth (target_z2) on the left wall,
    then uses Fermat's principle to find the exact path satisfying Snell's law 
    for all other segments.
    """
    def t_total(p):
        x1, z3, x4 = p
        # Path 1: Source to Layer 1 Interface
        t1 = np.sqrt((x1 - src_x)**2 + 10**2) / vp_layers[0]
        # Path 2: Layer 1 Interface to Forced Left Cave Wall Coordinate
        t2 = np.sqrt((-10 - x1)**2 + (target_z2 - 10)**2) / vp_layers[1]
        # Path 3: Transmission inside Cave
        t3 = np.sqrt((10 - (-10))**2 + (z3 - target_z2)**2) / cave_v
        # Path 4: Cave Right Wall to Layer 1 Interface
        t4 = np.sqrt((x4 - 10)**2 + (10 - z3)**2) / vp_layers[1]
        # Path 5: Layer 1 Interface to Surface Receiver
        t5 = np.sqrt((rec_x - x4)**2 + 10**2) / vp_layers[0]
        return t1 + t2 + t3 + t4 + t5

    x0 = [-60.0, target_z2, 40.0]
    bnds = [(src_x, -10.0), (cave_z1, cave_z2), (10.0, rec_x)]
    res = minimize(t_total, x0=x0, bounds=bnds)
    return res.x

# Calculate standard Head Wave parameters
p_head = 1.0 / vp_layers[2]
theta1 = np.arcsin(p_head * vp_layers[0])
theta2 = np.arcsin(p_head * vp_layers[1])
dx1 = thicknesses[0] * np.tan(theta1)
dx2 = thicknesses[1] * np.tan(theta2)
x_crit = dx1 + dx2

# =============================================================================
# 3. FIGURES 1 & 2: WITH CAVE ENVIRONMENT
# =============================================================================
fig1, ax1 = plt.subplots(figsize=(11, 6))

ax1.add_patch(patches.Rectangle((-150, 0), 300, 10, facecolor='#eaeaea', edgecolor='gray', linestyle='--'))
ax1.add_patch(patches.Rectangle((-150, 10), 300, 25, facecolor='#dcdcdc', edgecolor='gray', linestyle='--'))
ax1.add_patch(patches.Rectangle((-150, 35), 300, 45, facecolor='#c8c8c8', edgecolor='gray', linestyle='--'))
cave_patch = patches.Rectangle((cave_x1, cave_z1), cave_w, cave_h, facecolor='#6baed6', edgecolor='black', linewidth=2, hatch='//', label=f'Cave ({cave_v} m/s)')
ax1.add_patch(cave_patch)

ax1.text(-140, 5, f"Layer 1: {vp_layers[0]} m/s", weight='bold')
ax1.text(-140, 18, f"Layer 2: {vp_layers[1]} m/s", weight='bold')
ax1.text(-140, 55, f"Layer 3: {vp_layers[2]} m/s", weight='bold')
ax1.text(0, 22.5, f"Cave\n{cave_v} m/s", ha='center', va='center', weight='bold')

# RAY 1: Reflection (Crimson)
x_ref1 = solve_reflection_l1(src_x, rec_near)
ax1.plot([src_x, x_ref1, rec_near], [0.0, 10.0, 0.0], color='crimson', linewidth=2.5, label='Layer 1 Reflection')

# RAY 2: FORCE INTERSECTION AT MID-DEPTH (z=22.5) -> Shows true horizontal flattening
forced_z2 = 22.5
x1_c, z3_c, x4_c = solve_forced_mid_cave_ray(src_x, rec_far, forced_z2)
ray_c_x = [src_x, x1_c, cave_x1, cave_x2, x4_c, rec_far]
ray_c_y = [0.0, 10.0, forced_z2, z3_c, 10.0, 0.0]
ax1.plot(ray_c_x, ray_c_y, color='darkorange', linewidth=2.5, marker='o', label='Mid-Cave Forced Ray')

# RAY 3: Head Wave (Teal)
x_travel = (rec_far - src_x) - (2 * x_crit)
ray3_x = [src_x, src_x + dx1, src_x + x_crit, src_x + x_crit + x_travel, src_x + x_crit + x_travel + dx2, rec_far]
ray3_y = [0.0, 10.0, 35.0, 35.0, 10.0, 0.0]
ax1.plot(ray3_x, ray3_y, color='teal', linewidth=2, linestyle='--', label='Basement Head Wave')

ax1.plot(src_x, 0, marker='*', color='red', markersize=15, label='Source')
ax1.plot(rec_far, 0, marker='^', color='blue', markersize=10, label='Far Receiver')
ax1.set_xlim(-150, 150)
ax1.set_ylim(75, -2)
ax1.set_xlabel('Profile Position (m)', fontweight='bold')
ax1.set_ylabel('Depth (m)', fontweight='bold')
ax1.set_title("Figure 1: 2D Velocity Model with Forced Mid-Cave Snell Deflections", fontsize=12, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, linestyle=':', alpha=0.5)

# FIGURE 2: Travel-Time Curves (With Cave)
fig2, ax2 = plt.subplots(figsize=(11, 5))
t_direct, t_head = [], []
for rx in receivers:
    dist = np.abs(rx - src_x)
    t_direct.append(dist / vp_layers[0])
    if dist >= 2 * x_crit:
        t_base = (dist / vp_layers[2]) + 2 * (thicknesses[0]*np.cos(theta1)/vp_layers[0] + thicknesses[1]*np.cos(theta2)/vp_layers[1])
        if rx > cave_x2 + 25:
            t_base += cave_w * ((1.0 / cave_v) - (1.0 / vp_layers[1]))
        t_head.append(t_base)
    else:
        t_head.append(np.nan)

ax2.plot(receivers, np.array(t_direct)*1000, color='crimson', linewidth=2, label='Direct Wave')
ax2.plot(receivers, np.array(t_head)*1000, color='teal', linewidth=2.5, label='Layer 3 Head Wave')
ax2.axvspan(cave_x2 + 20, 150, color='darkorange', alpha=0.1, label='Cave Delay Shadow Zone')
ax2.set_xlim(-150, 150)
ax2.set_ylabel('Travel Time (ms)', fontweight='bold')
ax2.set_title("Figure 2: Travel-Time (t-x) Curves (With Cave Delay)", fontsize=12, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, linestyle=':')

# =============================================================================
# 4. FIGURES 3 & 4: REFERENCE BACKGROUND PLOTS (WITHOUT CAVE)
# =============================================================================
fig3, ax3 = plt.subplots(figsize=(11, 6))
ax3.add_patch(patches.Rectangle((-150, 0), 300, 10, facecolor='#eaeaea', edgecolor='gray', linestyle='--'))
ax3.add_patch(patches.Rectangle((-150, 10), 300, 25, facecolor='#dcdcdc', edgecolor='gray', linestyle='--'))
ax3.add_patch(patches.Rectangle((-150, 35), 300, 45, facecolor='#c8c8c8', edgecolor='gray', linestyle='--'))

ax3.text(-140, 5, f"Layer 1: {vp_layers[0]} m/s", weight='bold')
ax3.text(-140, 18, f"Layer 2: {vp_layers[1]} m/s", weight='bold')
ax3.text(-140, 55, f"Layer 3: {vp_layers[2]} m/s", weight='bold')

ax3.plot([src_x, x_ref1, rec_near], [0.0, 10.0, 0.0], color='crimson', linewidth=2.5, label='Layer 1 Reflection')
ax3.plot(ray3_x, ray3_y, color='teal', linewidth=2, linestyle='--', label='Basement Head Wave')

def solve_layer2_ray(src_x, rec_x, target_z):
    def t_l2(p):
        x1, x2 = p
        return (np.sqrt((x1 - src_x)**2 + 10**2)/vp_layers[0] + 
                np.sqrt((x2 - x1)**2 + (target_z - 10)**2 * 2)/vp_layers[1] + 
                np.sqrt((rec_x - x2)**2 + 10**2)/vp_layers[0])
    res = minimize(t_l2, x0=[-40, 40], bounds=[(src_x, 0), (0, rec_x)])
    return res.x

x1_noc, x2_noc = solve_layer2_ray(src_x, rec_far, forced_z2)
ax3.plot([src_x, x1_noc, 0.0, x2_noc, rec_far], [0.0, 10.0, forced_z2, 10.0, 0.0], color='darkorange', linewidth=2.5, label='Unperturbed Layer 2 Ray')

ax3.plot(src_x, 0, marker='*', color='red', markersize=15, label='Source')
ax3.plot(rec_far, 0, marker='^', color='blue', markersize=10, label='Far Receiver')
ax3.set_xlim(-150, 150)
ax3.set_ylim(75, -2)
ax3.set_xlabel('Profile Position (m)', fontweight='bold')
ax3.set_ylabel('Depth (m)', fontweight='bold')
ax3.set_title("Figure 3: Reference Model (Without Cave Anomaly)", fontsize=12, fontweight='bold')
ax3.legend(loc='lower right')
ax3.grid(True, linestyle=':', alpha=0.5)

# FIGURE 4: Reference Travel-Time Curves (Without Cave)
fig4, ax4 = plt.subplots(figsize=(11, 5))
t_head_nocave = []
for rx in receivers:
    dist = np.abs(rx - src_x)
    if dist >= 2 * x_crit:
        t_base = (dist / vp_layers[2]) + 2 * (thicknesses[0]*np.cos(theta1)/vp_layers[0] + thicknesses[1]*np.cos(theta2)/vp_layers[1])
        t_head_nocave.append(t_base)
    else:
        t_head_nocave.append(np.nan)

ax4.plot(receivers, np.array(t_direct)*1000, color='crimson', linewidth=2, label='Direct Wave')
ax4.plot(receivers, np.array(t_head_nocave)*1000, color='teal', linewidth=2.5, label='Layer 3 Head Wave (No Delay)')
ax4.set_xlim(-150, 150)
ax4.set_ylabel('Travel Time (ms)', fontweight='bold')
ax4.set_title("Figure 4: Reference Travel-Time Curves (Linear Baseline)", fontsize=12, fontweight='bold')
ax4.legend(loc='upper left')
ax4.grid(True, linestyle=':')

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.optimize import minimize

# =============================================================================
# 1. PHYSICAL MODEL PARAMETERS
# =============================================================================
vp_layers = [1600.0, 3600.0, 4750.0]  # Layer velocities (m/s)
thicknesses = [10.0, 25.0, 45.0]     # Layer thicknesses (m)
z_nodes = [0.0, 10.0, 35.0, 80.0]     # Interface depths (m)

# Cave parameters (Centered at x=0, z=22.5)
cave_v = 1500.0                       
cave_w, cave_h = 20.0, 20.0          
cave_x1, cave_x2 = -10.0, 10.0        
cave_z1, cave_z2 = 12.5, 32.5         

# Geometry setup
src_x = -150.0
rec_near = 40.0
rec_far = 140.0      # MODIFIED: Shifted receiver closer to center to force a mid-cave path
receivers = np.linspace(-140, 140, 561)

# =============================================================================
# 2. FIXED SNELL'S LAW SOLVERS VIA FERMAT'S PRINCIPLE
# =============================================================================
def solve_reflection_l1(src_x, rec_x):
    def t_field(x_int):
        return (np.sqrt((x_int - src_x)**2 + 10**2) + np.sqrt((rec_x - x_int)**2 + 10**2)) / vp_layers[0]
    res = minimize(t_field, x0=[(src_x + rec_x)/2], bounds=[(min(src_x, rec_x), max(src_x, rec_x))])
    return float(res.x[0]) # Extract scalar float directly

def solve_forced_mid_cave_ray(src_x, rec_x, target_z2):
    def t_total(p):
        x1, z3, x4 = p
        t1 = np.sqrt((x1 - src_x)**2 + 10**2) / vp_layers[0]
        t2 = np.sqrt((-10 - x1)**2 + (target_z2 - 10)**2) / vp_layers[1]
        t3 = np.sqrt((10 - (-10))**2 + (z3 - target_z2)**2) / cave_v
        t4 = np.sqrt((x4 - 10)**2 + (10 - z3)**2) / vp_layers[1]
        t5 = np.sqrt((rec_x - x4)**2 + 10**2) / vp_layers[0]
        return t1 + t2 + t3 + t4 + t5

    x0 = [-60.0, target_z2, 40.0]
    bnds = [(src_x, -10.0), (cave_z1, cave_z2), (10.0, rec_x)]
    res = minimize(t_total, x0=x0, bounds=bnds)
    return res.x # Returns array package: x1, z3, x4

# Calculate standard Head Wave parameters
p_head = 1.0 / vp_layers[2]
theta1 = np.arcsin(p_head * vp_layers[0])
theta2 = np.arcsin(p_head * vp_layers[1])
dx1 = thicknesses[0] * np.tan(theta1)
dx2 = thicknesses[1] * np.tan(theta2)
x_crit = dx1 + dx2

# =============================================================================
# 3. FIGURES 1 & 2: WITH CAVE ENVIRONMENT
# =============================================================================
fig1, ax1 = plt.subplots(figsize=(11, 6))

ax1.add_patch(patches.Rectangle((-150, 0), 300, 10, facecolor='#eaeaea', edgecolor='gray', linestyle='--'))
ax1.add_patch(patches.Rectangle((-150, 10), 300, 25, facecolor='#dcdcdc', edgecolor='gray', linestyle='--'))
ax1.add_patch(patches.Rectangle((-150, 35), 300, 45, facecolor='#c8c8c8', edgecolor='gray', linestyle='--'))
cave_patch = patches.Rectangle((cave_x1, cave_z1), cave_w, cave_h, facecolor='#6baed6', edgecolor='black', linewidth=2, hatch='//', label=f'Cave ({cave_v} m/s)')
ax1.add_patch(cave_patch)

ax1.text(-140, 5, f"Layer 1: {vp_layers[0]} m/s", weight='bold')
ax1.text(-140, 18, f"Layer 2: {vp_layers[1]} m/s", weight='bold')
ax1.text(-140, 55, f"Layer 3: {vp_layers[2]} m/s", weight='bold')
ax1.text(0, 22.5, f"Cave\n{cave_v} m/s", ha='center', va='center', weight='bold')

# RAY 1: Reflection (Crimson) - FIXED scalar placement
x_ref1 = solve_reflection_l1(src_x, rec_near)
ax1.plot([src_x, x_ref1, rec_near], [0.0, 10.0, 0.0], color='crimson', linewidth=2.5, label='Layer 1 Reflection')

# RAY 2: FORCE INTERSECTION AT MID-DEPTH (z=22.5) -> Shows true horizontal flattening
forced_z2 = 22.5
p_opt = solve_forced_mid_cave_ray(src_x, rec_far, forced_z2)
ray_c_x = [src_x, float(p_opt[0]), cave_x1, cave_x2, float(p_opt[2]), rec_far]
ray_c_y = [0.0, 10.0, forced_z2, float(p_opt[1]), 10.0, 0.0]
ax1.plot(ray_c_x, ray_c_y, color='darkorange', linewidth=2.5, marker='o', label='Mid-Cave Forced Ray')

# RAY 3: Head Wave (Teal)
x_travel = (rec_far - src_x) - (2 * x_crit)
ray3_x = [src_x, src_x + dx1, src_x + x_crit, src_x + x_crit + x_travel, src_x + x_crit + x_travel + dx2, rec_far]
ray3_y = [0.0, 10.0, 35.0, 35.0, 10.0, 0.0]
ax1.plot(ray3_x, ray3_y, color='teal', linewidth=2, linestyle='--', label='Basement Head Wave')

ax1.plot(src_x, 0, marker='*', color='red', markersize=15, label='Source')
ax1.plot(rec_far, 0, marker='^', color='blue', markersize=10, label='Far Receiver')
ax1.set_xlim(-150, 150)
ax1.set_ylim(75, -2)
ax1.set_xlabel('Profile Position (m)', fontweight='bold')
ax1.set_ylabel('Depth (m)', fontweight='bold')
ax1.set_title("Figure 1: 2D Velocity Model with Forced Mid-Cave Snell Deflections", fontsize=12, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, linestyle=':', alpha=0.5)

# FIGURE 2: Travel-Time Curves (With Cave)
fig2, ax2 = plt.subplots(figsize=(11, 5))
t_direct, t_head = [], []
for rx in receivers:
    dist = np.abs(rx - src_x)
    t_direct.append(dist / vp_layers[0])
    if dist >= 2 * x_crit:
        t_base = (dist / vp_layers[2]) + 2 * (thicknesses[0]*np.cos(theta1)/vp_layers[0] + thicknesses[1]*np.cos(theta2)/vp_layers[1])
        if rx > cave_x2 + 25:
            t_base += cave_w * ((1.0 / cave_v) - (1.0 / vp_layers[1]))
        t_head.append(t_base)
    else:
        t_head.append(np.nan)

ax2.plot(receivers, np.array(t_direct)*1000, color='crimson', linewidth=2, label='Direct Wave')
ax2.plot(receivers, np.array(t_head)*1000, color='teal', linewidth=2.5, label='Layer 3 Head Wave')
ax2.axvspan(cave_x2 + 20, 150, color='darkorange', alpha=0.1, label='Cave Delay Shadow Zone')
ax2.set_xlim(-150, 150)
ax2.set_ylabel('Travel Time (ms)', fontweight='bold')
ax2.set_title("Figure 2: Travel-Time (t-x) Curves (With Cave Delay)", fontsize=12, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, linestyle=':')

# =============================================================================
# 4. FIGURES 3 & 4: REFERENCE BACKGROUND PLOTS (WITHOUT CAVE)
# =============================================================================
fig3, ax3 = plt.subplots(figsize=(11, 6))
ax3.add_patch(patches.Rectangle((-150, 0), 300, 10, facecolor='#eaeaea', edgecolor='gray', linestyle='--'))
ax3.add_patch(patches.Rectangle((-150, 10), 300, 25, facecolor='#dcdcdc', edgecolor='gray', linestyle='--'))
ax3.add_patch(patches.Rectangle((-150, 35), 300, 45, facecolor='#c8c8c8', edgecolor='gray', linestyle='--'))

ax3.text(-140, 5, f"Layer 1: {vp_layers[0]} m/s", weight='bold')
ax3.text(-140, 18, f"Layer 2: {vp_layers[1]} m/s", weight='bold')
ax3.text(-140, 55, f"Layer 3: {vp_layers[2]} m/s", weight='bold')

ax3.plot([src_x, x_ref1, rec_near], [0.0, 10.0, 0.0], color='crimson', linewidth=2.5, label='Layer 1 Reflection')
ax3.plot(ray3_x, ray3_y, color='teal', linewidth=2, linestyle='--', label='Basement Head Wave')

def solve_layer2_ray(src_x, rec_x, target_z):
    def t_l2(p):
        x1, x2 = p
        return (np.sqrt((x1 - src_x)**2 + 10**2)/vp_layers[0] + 
                np.sqrt((x2 - x1)**2 + (target_z - 10)**2 * 2)/vp_layers[1] + 
                np.sqrt((rec_x - x2)**2 + 10**2)/vp_layers[0])
    res = minimize(t_l2, x0=[-40, 40], bounds=[(src_x, 0), (0, rec_x)])
    return res.x

p_noc = solve_layer2_ray(src_x, rec_far, forced_z2)
ax3.plot([src_x, float(p_noc[0]), 0.0, float(p_noc[1]), rec_far], [0.0, 10.0, forced_z2, 10.0, 0.0], color='darkorange', linewidth=2.5, label='Unperturbed Layer 2 Ray')

ax3.plot(src_x, 0, marker='*', color='red', markersize=15, label='Source')
ax3.plot(rec_far, 0, marker='^', color='blue', markersize=10, label='Far Receiver')
ax3.set_xlim(-150, 150)
ax3.set_ylim(75, -2)
ax3.set_xlabel('Profile Position (m)', fontweight='bold')
ax3.set_ylabel('Depth (m)', fontweight='bold')
ax3.set_title("Figure 3: Reference Model (Without Cave Anomaly)", fontsize=12, fontweight='bold')
ax3.legend(loc='lower right')
ax3.grid(True, linestyle=':', alpha=0.5)

# FIGURE 4: Reference Travel-Time Curves (Without Cave)
fig4, ax4 = plt.subplots(figsize=(11, 5))
t_head_nocave = []
for rx in receivers:
    dist = np.abs(rx - src_x)
    if dist >= 2 * x_crit:
        t_base = (dist / vp_layers[2]) + 2 * (thicknesses[0]*np.cos(theta1)/vp_layers[0] + thicknesses[1]*np.cos(theta2)/vp_layers[1])
        t_head_nocave.append(t_base)
    else:
        t_head_nocave.append(np.nan)

ax4.plot(receivers, np.array(t_direct)*1000, color='crimson', linewidth=2, label='Direct Wave')
ax4.plot(receivers, np.array(t_head_nocave)*1000, color='teal', linewidth=2.5, label='Layer 3 Head Wave (No Delay)')
ax4.set_xlim(-150, 150)
ax4.set_ylabel('Travel Time (ms)', fontweight='bold')
ax4.set_title("Figure 4: Reference Travel-Time Curves (Linear Baseline)", fontsize=12, fontweight='bold')
ax4.legend(loc='upper left')
ax4.grid(True, linestyle=':')

plt.show()


In [ ]:
The ValueError happens because scipy.optimize.minimize returns a 1-element NumPy array containing a scalar ([x_ref1]). When placed inside the list ([src_x, x_ref1, rec_near]), it builds an inhomogeneous, nested structure [scalar, array([scalar]), scalar] which breaks Matplotlib's 1D vector parser.
Extracting the pure float element using x_ref1[0] satisfies the list dimensions and resolves the array-element sequence conflict.
Here is the fully sanitized script. The extraction fix has been cleanly applied to all optimization outputs across both figure generation blocks.
## Complete Debugged Script

import numpy as npimport matplotlib.pyplot as pltimport matplotlib.patches as patchesfrom scipy.optimize import minimize
# =============================================================================# 1. PHYSICAL MODEL PARAMETERS# =============================================================================vp_layers = [1000.0, 3350.0, 4750.0]  # Layer velocities (m/s)thicknesses = [10.0, 25.0, 45.0]     # Layer thicknesses (m)z_nodes = [0.0, 10.0, 35.0, 80.0]     # Interface depths (m)
# Cave parameters (Centered at x=0, z=22.5)cave_v = 1500.0                       cave_w, cave_h = 20.0, 20.0          cave_x1, cave_x2 = -10.0, 10.0        cave_z1, cave_z2 = 12.5, 32.5         
# Geometry setupsrc_x = -120.0rec_near = 40.0rec_far = 75.0      receivers = np.linspace(-150, 150, 301)
# =============================================================================# 2. FIXED SNELL'S LAW SOLVERS VIA FERMAT'S PRINCIPLE# =============================================================================def solve_reflection_l1(src_x, rec_x):
    def t_field(x_int):
        return (np.sqrt((x_int - src_x)**2 + 10**2) + np.sqrt((rec_x - x_int)**2 + 10**2)) / vp_layers[0]
    res = minimize(t_field, x0=[(src_x + rec_x)/2], bounds=[(min(src_x, rec_x), max(src_x, rec_x))])
    return float(res.x[0]) # Extract scalar float directly
def solve_forced_mid_cave_ray(src_x, rec_x, target_z2):
    def t_total(p):
        x1, z3, x4 = p
        t1 = np.sqrt((x1 - src_x)**2 + 10**2) / vp_layers[0]
        t2 = np.sqrt((-10 - x1)**2 + (target_z2 - 10)**2) / vp_layers[1]
        t3 = np.sqrt((10 - (-10))**2 + (z3 - target_z2)**2) / cave_v
        t4 = np.sqrt((x4 - 10)**2 + (10 - z3)**2) / vp_layers[1]
        t5 = np.sqrt((rec_x - x4)**2 + 10**2) / vp_layers[0]
        return t1 + t2 + t3 + t4 + t5

    x0 = [-60.0, target_z2, 40.0]
    bnds = [(src_x, -10.0), (cave_z1, cave_z2), (10.0, rec_x)]
    res = minimize(t_total, x0=x0, bounds=bnds)
    return res.x # Returns array package: x1, z3, x4
# Calculate standard Head Wave parametersp_head = 1.0 / vp_layers[2]theta1 = np.arcsin(p_head * vp_layers[0])theta2 = np.arcsin(p_head * vp_layers[1])dx1 = thicknesses[0] * np.tan(theta1)dx2 = thicknesses[1] * np.tan(theta2)x_crit = dx1 + dx2
# =============================================================================# 3. FIGURES 1 & 2: WITH CAVE ENVIRONMENT# =============================================================================fig1, ax1 = plt.subplots(figsize=(11, 6))

ax1.add_patch(patches.Rectangle((-150, 0), 300, 10, facecolor='#eaeaea', edgecolor='gray', linestyle='--'))
ax1.add_patch(patches.Rectangle((-150, 10), 300, 25, facecolor='#dcdcdc', edgecolor='gray', linestyle='--'))
ax1.add_patch(patches.Rectangle((-150, 35), 300, 45, facecolor='#c8c8c8', edgecolor='gray', linestyle='--'))cave_patch = patches.Rectangle((cave_x1, cave_z1), cave_w, cave_h, facecolor='#6baed6', edgecolor='black', linewidth=2, hatch='//', label=f'Cave ({cave_v} m/s)')
ax1.add_patch(cave_patch)

ax1.text(-140, 5, f"Layer 1: {vp_layers[0]} m/s", weight='bold')
ax1.text(-140, 18, f"Layer 2: {vp_layers[1]} m/s", weight='bold')
ax1.text(-140, 55, f"Layer 3: {vp_layers[2]} m/s", weight='bold')
ax1.text(0, 22.5, f"Cave\n{cave_v} m/s", ha='center', va='center', weight='bold')
# RAY 1: Reflection (Crimson) - FIXED scalar placementx_ref1 = solve_reflection_l1(src_x, rec_near)
ax1.plot([src_x, x_ref1, rec_near], [0.0, 10.0, 0.0], color='crimson', linewidth=2.5, label='Layer 1 Reflection')
# RAY 2: FORCE INTERSECTION AT MID-DEPTH (z=22.5) -> Shows true horizontal flatteningforced_z2 = 22.5p_opt = solve_forced_mid_cave_ray(src_x, rec_far, forced_z2)ray_c_x = [src_x, float(p_opt[0]), cave_x1, cave_x2, float(p_opt[2]), rec_far]ray_c_y = [0.0, 10.0, forced_z2, float(p_opt[1]), 10.0, 0.0]
ax1.plot(ray_c_x, ray_c_y, color='darkorange', linewidth=2.5, marker='o', label='Mid-Cave Forced Ray')
# RAY 3: Head Wave (Teal)x_travel = (rec_far - src_x) - (2 * x_crit)ray3_x = [src_x, src_x + dx1, src_x + x_crit, src_x + x_crit + x_travel, src_x + x_crit + x_travel + dx2, rec_far]ray3_y = [0.0, 10.0, 35.0, 35.0, 10.0, 0.0]
ax1.plot(ray3_x, ray3_y, color='teal', linewidth=2, linestyle='--', label='Basement Head Wave')

ax1.plot(src_x, 0, marker='*', color='red', markersize=15, label='Source')
ax1.plot(rec_far, 0, marker='^', color='blue', markersize=10, label='Far Receiver')
ax1.set_xlim(-150, 150)
ax1.set_ylim(75, -2)
ax1.set_xlabel('Profile Position (m)', fontweight='bold')
ax1.set_ylabel('Depth (m)', fontweight='bold')
ax1.set_title("Figure 1: 2D Velocity Model with Forced Mid-Cave Snell Deflections", fontsize=12, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, linestyle=':', alpha=0.5)
# FIGURE 2: Travel-Time Curves (With Cave)fig2, ax2 = plt.subplots(figsize=(11, 5))t_direct, t_head = [], []for rx in receivers:
    dist = np.abs(rx - src_x)
    t_direct.append(dist / vp_layers[0])
    if dist >= 2 * x_crit:
        t_base = (dist / vp_layers[2]) + 2 * (thicknesses[0]*np.cos(theta1)/vp_layers[0] + thicknesses[1]*np.cos(theta2)/vp_layers[1])
        if rx > cave_x2 + 25:
            t_base += cave_w * ((1.0 / cave_v) - (1.0 / vp_layers[1]))
        t_head.append(t_base)
    else:
        t_head.append(np.nan)

ax2.plot(receivers, np.array(t_direct)*1000, color='crimson', linewidth=2, label='Direct Wave')
ax2.plot(receivers, np.array(t_head)*1000, color='teal', linewidth=2.5, label='Layer 3 Head Wave')
ax2.axvspan(cave_x2 + 20, 150, color='darkorange', alpha=0.1, label='Cave Delay Shadow Zone')
ax2.set_xlim(-150, 150)
ax2.set_ylabel('Travel Time (ms)', fontweight='bold')
ax2.set_title("Figure 2: Travel-Time (t-x) Curves (With Cave Delay)", fontsize=12, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, linestyle=':')
# =============================================================================# 4. FIGURES 3 & 4: REFERENCE BACKGROUND PLOTS (WITHOUT CAVE)# =============================================================================fig3, ax3 = plt.subplots(figsize=(11, 6))
ax3.add_patch(patches.Rectangle((-150, 0), 300, 10, facecolor='#eaeaea', edgecolor='gray', linestyle='--'))
ax3.add_patch(patches.Rectangle((-150, 10), 300, 25, facecolor='#dcdcdc', edgecolor='gray', linestyle='--'))
ax3.add_patch(patches.Rectangle((-150, 35), 300, 45, facecolor='#c8c8c8', edgecolor='gray', linestyle='--'))

ax3.text(-140, 5, f"Layer 1: {vp_layers[0]} m/s", weight='bold')
ax3.text(-140, 18, f"Layer 2: {vp_layers[1]} m/s", weight='bold')
ax3.text(-140, 55, f"Layer 3: {vp_layers[2]} m/s", weight='bold')

ax3.plot([src_x, x_ref1, rec_near], [0.0, 10.0, 0.0], color='crimson', linewidth=2.5, label='Layer 1 Reflection')
ax3.plot(ray3_x, ray3_y, color='teal', linewidth=2, linestyle='--', label='Basement Head Wave')
def solve_layer2_ray(src_x, rec_x, target_z):
    def t_l2(p):
        x1, x2 = p
        return (np.sqrt((x1 - src_x)**2 + 10**2)/vp_layers[0] + 
                np.sqrt((x2 - x1)**2 + (target_z - 10)**2 * 2)/vp_layers[1] + 
                np.sqrt((rec_x - x2)**2 + 10**2)/vp_layers[0])
    res = minimize(t_l2, x0=[-40, 40], bounds=[(src_x, 0), (0, rec_x)])
    return res.x
p_noc = solve_layer2_ray(src_x, rec_far, forced_z2)
ax3.plot([src_x, float(p_noc[0]), 0.0, float(p_noc[1]), rec_far], [0.0, 10.0, forced_z2, 10.0, 0.0], color='darkorange', linewidth=2.5, label='Unperturbed Layer 2 Ray')

ax3.plot(src_x, 0, marker='*', color='red', markersize=15, label='Source')
ax3.plot(rec_far, 0, marker='^', color='blue', markersize=10, label='Far Receiver')
ax3.set_xlim(-150, 150)
ax3.set_ylim(75, -2)
ax3.set_xlabel('Profile Position (m)', fontweight='bold')
ax3.set_ylabel('Depth (m)', fontweight='bold')
ax3.set_title("Figure 3: Reference Model (Without Cave Anomaly)", fontsize=12, fontweight='bold')
ax3.legend(loc='lower right')
ax3.grid(True, linestyle=':', alpha=0.5)
# FIGURE 4: Reference Travel-Time Curves (Without Cave)fig4, ax4 = plt.subplots(figsize=(11, 5))t_head_nocave = []for rx in receivers:
    dist = np.abs(rx - src_x)
    if dist >= 2 * x_crit:
        t_base = (dist / vp_layers[2]) + 2 * (thicknesses[0]*np.cos(theta1)/vp_layers[0] + thicknesses[1]*np.cos(theta2)/vp_layers[1])
        t_head_nocave.append(t_base)
    else:
        t_head_nocave.append(np.nan)

ax4.plot(receivers, np.array(t_direct)*1000, color='crimson', linewidth=2, label='Direct Wave')
ax4.plot(receivers, np.array(t_head_nocave)*1000, color='teal', linewidth=2.5, label='Layer 3 Head Wave (No Delay)')
ax4.set_xlim(-150, 150)
ax4.set_ylabel('Travel Time (ms)', fontweight='bold')
ax4.set_title("Figure 4: Reference Travel-Time Curves (Linear Baseline)", fontsize=12, fontweight='bold')
ax4.legend(loc='upper left')
ax4.grid(True, linestyle=':')

plt.show()

If you are interested, let me know:

* Do you want to calculate reflection hyperbola equations from the top boundary of the cave ($z=12.5\text{ m}$)?
* Should we integrate a sliding source routine to show how the structural delay changes as the source moves across the layout?


